# TP 2 · Couche Silver avec Spark et Apache Iceberg

## Étape 2 · Créer la table Silver et écrire les données nettoyées

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("tp2").getOrCreate()

# Utilitaire : relève fichiers / snapshots / lignes après chaque étape
results = []
def stats(label):
    nb_f = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.files").collect()[0].c
    nb_s = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.snapshots").collect()[0].c
    nb_l = spark.table("demo.silver.orders").count()
    results.append((label, nb_f, nb_s, nb_l))
    print(f"[{label}] fichiers: {nb_f} | snapshots: {nb_s} | lignes: {nb_l}")

### Cellule 1 · Espace de noms et table

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.orders (
  order_id STRING, customer_id BIGINT, order_ts TIMESTAMP,
  city STRING, amount DECIMAL(10,2), status STRING,
  _ingested_at TIMESTAMP)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES ('write.parquet.compression-codec' = 'zstd')
""")

### Cellule 2 · Nettoyer et écrire

In [ ]:
raw = spark.read.parquet("/home/iceberg/notebooks/data/orders/")
silver = (raw
    .dropDuplicates(["order_id"])                       # unicité
    .filter(F.col("amount") > 0)                        # validité
    .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
    .withColumn("city", F.initcap(F.trim("city")))      # normalisation
    .withColumn("status", F.upper("status"))
    .drop("_batch_id"))
silver.writeTo("demo.silver.orders").append()
spark.table("demo.silver.orders").groupBy("status").count().show()

### Cellule 3 · Tables de métadonnées

In [ ]:
spark.sql("SELECT * FROM demo.silver.orders.snapshots").show(truncate=False)
spark.sql("""SELECT partition, record_count, file_size_in_bytes
             FROM demo.silver.orders.files LIMIT 10""").show(truncate=False)
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
stats("Étape 2 : après écriture Silver")

### Questions 1 à 3

In [ ]:
# Q1 : nombre de partitions (jours) distinctes
spark.sql("SELECT count(*) AS nb_partitions FROM demo.silver.orders.partitions").show()

In [ ]:
# Q2 : contenu du metadata.json courant (via le catalogue REST)
import requests, json
try:
    r = requests.get("http://rest:8181/v1/namespaces/silver/tables/orders").json()
    m = r["metadata"]
    print("fichier metadata :", r["metadata-location"])
    print("snapshot courant :", m["current-snapshot-id"])
    print("schéma           :", [(f["id"], f["name"], f["type"]) for f in m["schemas"][-1]["fields"]])
    print("partitionnement  :", json.dumps(m["partition-specs"][-1]))
except Exception as e:
    print("Lecture via REST impossible :", e)
    print("-> télécharger le dernier *.metadata.json depuis la console MinIO (warehouse/silver/orders/metadata/)")

In [ ]:
# Q3 : plan d'exécution avec filtre sur order_ts (colonne source)
plan = spark.sql("""EXPLAIN SELECT count(*) FROM demo.silver.orders
                    WHERE order_ts >= current_date() - INTERVAL 7 DAYS""").collect()[0][0]
print(plan)

## Étape 3 · Upsert avec MERGE INTO

In [ ]:
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").show(truncate=False)

def make_updates():
    """1 000 commandes PAID -> SHIPPED + 500 nouvelles commandes (mis en cache pour figer uuid())"""
    df = spark.sql("""
        SELECT * FROM (
          SELECT order_id, customer_id, order_ts, city, amount,
                 'SHIPPED' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders WHERE status = 'PAID' LIMIT 1000)
        UNION ALL
        SELECT * FROM (
          SELECT uuid() AS order_id, customer_id, current_timestamp() AS order_ts, city, amount,
                 'PAID' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders LIMIT 500)
    """).cache()
    df.count()
    df.createOrReplaceTempView("updates")
    return df

MERGE_SQL = """
MERGE INTO demo.silver.orders t
USING updates s
ON t.order_id = s.order_id
WHEN MATCHED AND t.status <> s.status THEN
  UPDATE SET t.status = s.status, t._ingested_at = s._ingested_at
WHEN NOT MATCHED THEN INSERT *
"""

### MERGE en copy-on-write (mode par défaut)

In [ ]:
upd1 = make_updates()
t0 = time.time()
spark.sql(MERGE_SQL)
t_cow = time.time() - t0
print(f"MERGE copy-on-write : {t_cow:.1f} s")
stats("Étape 3 : après MERGE copy-on-write")
spark.sql("""SELECT snapshot_id, committed_at, operation,
                    summary['added-records'], summary['deleted-records'],
                    summary['added-data-files'], summary['deleted-data-files']
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)

### Expérience : merge-on-read

In [ ]:
spark.sql("""ALTER TABLE demo.silver.orders SET TBLPROPERTIES (
  'write.merge.mode' = 'merge-on-read',
  'write.update.mode' = 'merge-on-read',
  'write.delete.mode' = 'merge-on-read')""")
upd2 = make_updates()          # nouveau lot : 1 000 autres PAID + 500 nouvelles commandes
t0 = time.time()
spark.sql(MERGE_SQL)
t_mor = time.time() - t0
print(f"MERGE merge-on-read : {t_mor:.1f} s")
stats("Étape 3 : après MERGE merge-on-read")
spark.sql("""SELECT snapshot_id, operation,
                    summary['added-data-files'] AS added_data, summary['deleted-data-files'] AS deleted_data,
                    summary['added-delete-files'] AS added_delete_files,
                    summary['added-position-deletes'] AS pos_deletes
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)
spark.sql("SELECT content, count(*) AS nb FROM demo.silver.orders.files GROUP BY content").show()

### Idempotence : rejouer le MERGE avec la même vue

In [ ]:
before = spark.table("demo.silver.orders").count()
spark.sql(MERGE_SQL)          # même vue 'updates' (upd2, en cache)
after = spark.table("demo.silver.orders").count()
print("lignes avant :", before, "| après :", after)
stats("Étape 3 : après re-MERGE identique")
spark.sql("""SELECT snapshot_id, operation, summary['added-records'], summary['changed-partition-count']
             FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1""").show(truncate=False)

## Étape 4 · Time travel, rollback et évolution du schéma

In [ ]:
snaps = spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").collect()
for s in snaps: print(s.snapshot_id, s.committed_at, s.operation)
snap_before_merge = snaps[0].snapshot_id          # l'append initial, avant tout MERGE
ts_before_merge = snaps[0].committed_at
print("snapshot avant MERGE :", snap_before_merge)

In [ ]:
print("--- état AVANT le MERGE (VERSION AS OF) ---")
spark.sql(f"SELECT status, count(*) AS nb FROM demo.silver.orders VERSION AS OF {snap_before_merge} GROUP BY status ORDER BY status").show()
print("--- état ACTUEL ---")
spark.sql("SELECT status, count(*) AS nb FROM demo.silver.orders GROUP BY status ORDER BY status").show()
spark.sql(f"SELECT count(*) AS nb FROM demo.silver.orders TIMESTAMP AS OF '{ts_before_merge}'").show()

### Erreur simulée puis rollback

In [ ]:
snap_before_delete = spark.sql("SELECT snapshot_id FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0].snapshot_id
n_before = spark.table("demo.silver.orders").count()
spark.sql("DELETE FROM demo.silver.orders WHERE status = 'SHIPPED'")
n_deleted = spark.table("demo.silver.orders").count()
print("lignes avant DELETE :", n_before, "| après DELETE :", n_deleted)
spark.sql(f"CALL demo.system.rollback_to_snapshot('silver.orders', {snap_before_delete})").show()
print("lignes après rollback :", spark.table("demo.silver.orders").count())
stats("Étape 4 : après DELETE + rollback")

### Évolution du schéma (sans réécriture)

In [ ]:
spark.sql("ALTER TABLE demo.silver.orders ADD COLUMN channel STRING")
spark.sql("ALTER TABLE demo.silver.orders RENAME COLUMN city TO delivery_city")
spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(12,2)")
spark.sql("DESCRIBE TABLE demo.silver.orders").show(truncate=False)
spark.sql("SELECT order_id, delivery_city, amount, channel FROM demo.silver.orders LIMIT 5").show()
stats("Étape 4 : après évolution du schéma")

In [ ]:
# Test : réduire DECIMAL(12,2) -> DECIMAL(10,2) doit être refusé
try:
    spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(10,2)")
except Exception as e:
    print("Refusé comme prévu :", str(e)[:300])

### Évolution du partitionnement : jour -> mois

In [ ]:
spark.sql("ALTER TABLE demo.silver.orders REPLACE PARTITION FIELD order_ts_day WITH months(order_ts)")
spark.sql("SELECT * FROM demo.silver.orders.partitions").show(truncate=False)

In [ ]:
# Petit lot de test pour voir où vont les nouvelles données
spark.sql("""INSERT INTO demo.silver.orders
             SELECT uuid(), 1, current_timestamp(), 'Test-Ville', CAST(10.00 AS DECIMAL(12,2)), 'PAID', current_timestamp(), 'web'""")
spark.sql("SELECT partition, record_count FROM demo.silver.orders.files ORDER BY partition DESC LIMIT 5").show(truncate=False)
stats("Étape 4 : après changement de partitionnement")

## Étape 5 · Maintenance et export

In [ ]:
spark.sql("""CALL demo.system.rewrite_data_files(
  table => 'silver.orders',
  options => map('target-file-size-bytes', '134217728'))""").show(truncate=False)
stats("Étape 5 : après compaction")

In [ ]:
spark.sql("""CALL demo.system.expire_snapshots(
  table => 'silver.orders',
  older_than => current_timestamp(),
  retain_last => 5)""").show(truncate=False)
stats("Étape 5 : après expire_snapshots")

In [ ]:
spark.sql("CALL demo.system.remove_orphan_files(table => 'silver.orders')").show(truncate=False)
stats("Étape 5 : après remove_orphan_files")

In [ ]:
import os
os.makedirs("/home/iceberg/notebooks/data/silver", exist_ok=True)
df = spark.table("demo.silver.orders").toPandas()
df.to_parquet("/home/iceberg/notebooks/data/silver/orders.parquet", index=False)
print(len(df), "lignes exportées")

## Récapitulatif (à recopier dans TP2.md)

In [ ]:
import pandas as pd
print(pd.DataFrame(results, columns=["étape", "fichiers", "snapshots", "lignes"]).to_string(index=False))
print(f"\nMERGE copy-on-write : {t_cow:.1f} s | merge-on-read : {t_mor:.1f} s")